# Task 3 - Data Quality Investigation

Profiles the source files and writes `03_dq_report.xlsx`.

Put this notebook in `03_dq_report/`. It reads `../data/`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

STATUSES = {"CAPTURED", "DECLINED", "FAILED", "REVERSED", "PENDING"}

DATA_DIR = "../data"
OUT_FILE = "03_dq_report.xlsx"

## Load

In [2]:
def load(data_dir):
    d = Path(data_dir)
    t = pd.read_csv(d / "transaction.csv", parse_dates=["created_at"])
    e = pd.read_csv(d / "payment_event.csv", parse_dates=["event_time", "ingestion_time"])
    f = pd.read_csv(d / "fraud_decision.csv")
    s = pd.read_csv(d / "settlement.csv", parse_dates=["settlement_date"])
    c = pd.read_csv(d / "chargeback.csv", parse_dates=["opened_at", "resolved_at"])
    fx = pd.read_csv(d / "fx_rate.csv", parse_dates=["rate_date"])
    rc = pd.read_csv(d / "route_cost.csv", parse_dates=["rate_date"])
    m = pd.read_csv(d / "merchant.csv")
    cu = pd.read_csv(d / "customer.csv", parse_dates=["onboarding_date"])
    rs = pd.read_csv(d / "merchant_risk_snapshot.csv")
    return dict(transaction=t, payment_event=e, fraud_decision=f, settlement=s,
                chargeback=c, fx_rate=fx, route_cost=rc, merchant=m,
                customer=cu, merchant_risk_snapshot=rs)

D = load(DATA_DIR)
for name, df in D.items():
    print(f"{name:<24} {len(df):>9,} rows")

transaction                260,287 rows
payment_event            1,000,316 rows
fraud_decision             466,451 rows
settlement                 224,242 rows
chargeback                     721 rows
fx_rate                      2,920 rows
route_cost                   3,285 rows
merchant                       600 rows
customer                    15,000 rows
merchant_risk_snapshot       7,200 rows


## Scorecard

One row per check across the six required dimensions. `severity` is the potential
impact of the check; `result` is what actually happened. A Blocking check that
passes is a control that held, not an issue.

In [3]:
def build_scorecard(D):
    t, e, f = D["transaction"], D["payment_event"], D["fraud_decision"]
    s, c, fx = D["settlement"], D["chargeback"], D["fx_rate"]
    rc, m, rs = D["route_cost"], D["merchant"], D["merchant_risk_snapshot"]

    rows = []

    def add(dim, check, table, failed, total, sev, rem):
        rows.append({
            "dimension": dim, "check_name": check, "table_name": table,
            "failed_rows": int(failed), "total_rows": int(total),
            "failed_pct": round(100.0 * failed / total, 3) if total else 0.0,
            "severity": sev, "remediation": rem,
        })

    nt = len(t)

    # ---------------------------------------------------------- completeness
    add("Completeness", "transaction.merchant_id is null", "transaction",
        t.merchant_id.isna().sum(), nt, "Warning",
        "Retain with flag; exclude only from merchant-level breakdowns")
    add("Completeness", "transaction.customer_id is null", "transaction",
        t.customer_id.isna().sum(), nt, "Informational", "Retain")
    add("Completeness", "transaction.amount is null", "transaction",
        t.amount.isna().sum(), nt, "Blocking", "Quarantine - cannot compute value")
    add("Completeness", "chargeback.resolved_at is null (still open)", "chargeback",
        c.resolved_at.isna().sum(), len(c), "Informational",
        "Retain - open cases are a valid business state")

    # -------------------------------------------------------------- validity
    add("Validity", "fraud_decision.risk_score outside 0 to 1", "fraud_decision",
        ((f.risk_score < 0) | (f.risk_score > 1)).sum(), len(f), "Blocking",
        "Repair - normalise by model_version before any threshold comparison")
    add("Validity", "transaction.amount < 0 with status REVERSED (legitimate)", "transaction",
        ((t.amount < 0) & (t.status == "REVERSED")).sum(), nt, "Informational",
        "Retain - genuine reversals, net them rather than dropping")
    add("Validity", "transaction.amount < 0 with status CAPTURED (impossible)", "transaction",
        ((t.amount < 0) & (t.status == "CAPTURED")).sum(), nt, "Blocking",
        "Quarantine - a captured payment cannot carry a negative amount")
    add("Validity", "transaction.amount = 0", "transaction",
        (t.amount == 0).sum(), nt, "Warning",
        "Quarantine - zero-value attempts carry no economics")
    add("Validity", "transaction.status outside the known set", "transaction",
        (~t.status.isin(STATUSES)).sum(), nt, "Blocking", "Quarantine")
    add("Validity", "fx_rate.rate_to_usd not positive", "fx_rate",
        (fx.rate_to_usd <= 0).sum(), len(fx), "Blocking", "Repair from source")
    add("Validity", "route_cost.variable_fee_pct outside 0 to 0.10", "route_cost",
        ((rc.variable_fee_pct < 0) | (rc.variable_fee_pct > 0.10)).sum(), len(rc),
        "Blocking", "Repair from source")

    # ------------------------------------------------------------ uniqueness
    add("Uniqueness", "transaction_id is not unique", "transaction",
        nt - t.transaction_id.nunique(), nt, "Blocking",
        "Deduplicate - the declared primary key must hold")
    per_txn = s.groupby("transaction_id").size()
    add("Uniqueness", "more than one settlement row per transaction", "settlement",
        (per_txn > 1).sum(), len(s), "Blocking",
        "Sum fee_amount to transaction grain before joining")
    add("Uniqueness", "merchant_risk_snapshot composite key is not unique",
        "merchant_risk_snapshot",
        len(rs) - rs.duplicated(["merchant_id", "snapshot_month"]).eq(False).sum(),
        len(rs), "Blocking", "Deduplicate")
    add("Uniqueness", "fx_rate composite key is not unique", "fx_rate",
        fx.duplicated(["rate_date", "currency", "rate_type"]).sum(), len(fx),
        "Blocking", "Deduplicate")

    # ----------------------------------------------------------- consistency
    known_m = set(m.merchant_id)
    add("Consistency", "transaction.merchant_id not found in merchant", "transaction",
        (t.merchant_id.notna() & ~t.merchant_id.isin(known_m)).sum(), nt,
        "Blocking", "Quarantine - orphaned merchant reference")

    known_t = set(t.transaction_id)
    add("Consistency", "settlement.transaction_id not found in transaction", "settlement",
        (~s.transaction_id.isin(known_t)).sum(), len(s), "Blocking",
        "Exclude and report as a reconciliation break to Finance Operations")
    add("Consistency", "chargeback.transaction_id not found in transaction", "chargeback",
        (~c.transaction_id.isin(known_t)).sum(), len(c), "Blocking", "Exclude")

    status_of = t.set_index("transaction_id").status
    cb_status = status_of.reindex(c.transaction_id)
    add("Consistency", "chargeback raised against a non-captured transaction", "chargeback",
        (cb_status.notna() & (cb_status != "CAPTURED")).sum(), len(c), "Blocking",
        "Quarantine - a payment that never captured cannot be charged back")

    has_created = set(e.loc[e.event_type == "CREATED", "transaction_id"])
    add("Consistency", "transaction has no CREATED lifecycle event", "transaction",
        (~t.transaction_id.isin(has_created)).sum(), nt, "Warning",
        "Retain with flag - event pipeline gap")

    # ------------------------------------------------------------ timeliness
    lag_h = (e.ingestion_time - e.event_time).dt.total_seconds() / 3600
    add("Timeliness", "event ingested more than 2 days after it occurred", "payment_event",
        (lag_h > 48).sum(), len(e), "Warning",
        "Retain - use event_time for KPI periods, ingestion_time for lag reporting")

    created_of = t.set_index("transaction_id").created_at
    ev_created = created_of.reindex(e.transaction_id).to_numpy()
    add("Timeliness", "event_time earlier than the parent transaction created_at",
        "payment_event", (e.event_time.to_numpy() < ev_created).sum(), len(e),
        "Blocking", "Quarantine - an event cannot precede its transaction")

    cb_created = created_of.reindex(c.transaction_id).to_numpy()
    add("Timeliness", "chargeback opened before the transaction occurred", "chargeback",
        (c.opened_at.to_numpy() < cb_created).sum(), len(c), "Blocking", "Quarantine")

    st_created = created_of.reindex(s.transaction_id)
    add("Timeliness", "settlement dated before the transaction occurred", "settlement",
        (s.settlement_date.to_numpy() < st_created.dt.normalize().to_numpy()).sum(),
        len(s), "Blocking", "Quarantine")

    # -------------------------------------------------------- reconciliation
    cap = t[t.status == "CAPTURED"]
    settled_ids = set(s.transaction_id)
    add("Reconciliation", "captured transaction with no settlement row", "transaction",
        (~cap.transaction_id.isin(settled_ids)).sum(), len(cap), "Blocking",
        "Retain with flag and impute the fee from route_cost; excluding them biases recent months")

    fx_set = fx[fx.rate_type == "SETTLEMENT"].set_index(["currency", "rate_date"]).rate_to_usd
    j = s.merge(t[["transaction_id", "amount", "currency", "created_at"]],
                on="transaction_id", how="inner")
    key = pd.MultiIndex.from_arrays([j.currency, j.created_at.dt.normalize()])
    rate = fx_set.reindex(key).to_numpy()
    expected_usd = j.amount.to_numpy() * rate
    actual_usd = j.settlement_amount.to_numpy() + j.fee_amount.to_numpy()
    mismatch = np.abs(actual_usd - expected_usd) > 0.01
    add("Reconciliation", "settlement value does not match transaction value in USD",
        "settlement", np.nansum(mismatch), len(j), "Blocking",
        "Sum settlement rows to transaction grain before reconciling")

    sc = pd.DataFrame(rows)
    sc.insert(6, "result", np.where(sc.failed_rows > 0, "FAIL", "PASS"))
    rank = {"Blocking": 0, "Warning": 1, "Informational": 2}
    sc["_r"] = sc.severity.map(rank)
    return (sc.sort_values(["_r", "failed_pct"], ascending=[True, False])
              .drop(columns="_r").reset_index(drop=True))

scorecard = build_scorecard(D)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 46)
scorecard[["dimension", "check_name", "failed_rows", "total_rows", "failed_pct",
           "result", "severity"]]

,dimension,check_name,failed_rows,total_rows,failed_pct,result,severity
0,Validity,fraud_decision.risk_score outside 0 to 1,176568,466451,37.853,FAIL,Blocking
1,Reconciliation,captured transaction with no settlement row,11137,233598,4.768,FAIL,Blocking
2,Reconciliation,settlement value does not match transactio...,3038,223795,1.357,FAIL,Blocking
3,Uniqueness,more than one settlement row per transaction,1334,224242,0.595,FAIL,Blocking
4,Consistency,settlement.transaction_id not found in tra...,447,224242,0.199,FAIL,Blocking
5,Validity,transaction.amount < 0 with status CAPTURE...,390,260287,0.150,FAIL,Blocking
6,Completeness,transaction.amount is null,0,260287,0.000,PASS,Blocking
7,Validity,transaction.status outside the known set,0,260287,0.000,PASS,Blocking
8,Validity,fx_rate.rate_to_usd not positive,0,2920,0.000,PASS,Blocking
9,Validity,route_cost.variable_fee_pct outside 0 to 0.10,0,3285,0.000,PASS,Blocking


## Detail profiles

A defect rate that is not uniform across a dimension is a finding in itself.
The reconciliation breakdown decomposes the USD mismatch to a zero residual -
proving the break is fully attributed to defects already identified.

In [4]:
def build_details(D):
    t, e, f = D["transaction"], D["payment_event"], D["fraud_decision"]
    s, rc, rs = D["settlement"], D["route_cost"], D["merchant_risk_snapshot"]
    out = {}

    g = t.groupby("channel").agg(attempts=("transaction_id", "count"),
                                 missing_merchant=("merchant_id", lambda x: x.isna().sum()))
    g["missing_pct"] = (100 * g.missing_merchant / g.attempts).round(3)
    out["Missing merchant by channel"] = g.sort_values("missing_pct", ascending=False).reset_index()

    g = f.groupby("model_version").agg(decisions=("risk_score", "count"),
                                       min_score=("risk_score", "min"),
                                       max_score=("risk_score", "max"))
    g["scores_above_one"] = f[f.risk_score > 1].groupby("model_version").size().reindex(g.index).fillna(0).astype(int)
    g["above_one_pct"] = (100 * g.scores_above_one / g.decisions).round(2)
    g[["min_score", "max_score"]] = g[["min_score", "max_score"]].round(4)
    out["Risk score by model version"] = g.reset_index()[
        ["model_version", "decisions", "min_score", "max_score", "scores_above_one", "above_one_pct"]]

    lag_h = (e.ingestion_time - e.event_time).dt.total_seconds() / 3600
    tmp = pd.DataFrame({"event_month": e.event_time.dt.to_period("M").astype(str),
                        "late": (lag_h > 48).astype(int), "lag_h": lag_h})
    g = tmp.groupby("event_month").agg(events=("late", "count"),
                                       late_over_2d=("late", "sum"),
                                       max_lag_days=("lag_h", "max"))
    g["late_pct"] = (100 * g.late_over_2d / g.events).round(2)
    g["max_lag_days"] = (g.max_lag_days / 24).round(1)
    out["Event lag by month"] = g.reset_index()[
        ["event_month", "events", "late_over_2d", "late_pct", "max_lag_days"]]

    cap = t[t.status == "CAPTURED"].copy()
    cap["txn_month"] = cap.created_at.dt.to_period("M").astype(str)
    cap["has_settlement"] = cap.transaction_id.isin(set(s.transaction_id))
    g = cap.groupby("txn_month").agg(captured=("transaction_id", "count"),
                                     settled=("has_settlement", "sum"))
    g["no_settlement"] = g.captured - g.settled
    g["no_settlement_pct"] = (100 * g.no_settlement / g.captured).round(2)
    out["Settlement coverage by month"] = g.reset_index()[
        ["txn_month", "captured", "no_settlement", "no_settlement_pct"]]

    routes = rc[["route_id", "provider"]].drop_duplicates()
    ev = e[["transaction_id", "processing_ms"]].merge(
        t[["transaction_id", "route_id", "created_at"]], on="transaction_id")
    ev = ev.merge(routes, on="route_id")
    ev["txn_month"] = ev.created_at.dt.to_period("M").astype(str)
    g = (ev.groupby(["provider", "txn_month"])
           .agg(attempts=("processing_ms", "count"),
                p50_ms=("processing_ms", "median"),
                p95_ms=("processing_ms", lambda x: x.quantile(0.95)))
           .round(0).reset_index())
    out["Latency by provider month"] = g

    base_gpv = cap.amount.sum()
    naive_rs = cap.merge(rs[["merchant_id", "snapshot_month"]], on="merchant_id")
    naive_fr = cap.merge(f[["transaction_id", "risk_score"]], on="transaction_id")
    agg_fr = f.groupby("transaction_id").risk_score.max().rename("max_score").reset_index()
    good_fr = cap.merge(agg_fr, on="transaction_id")
    out["Fan-out proof"] = pd.DataFrame([
        ("correct: no join", len(cap), base_gpv, 1.0),
        ("naive join to risk snapshot", len(naive_rs), naive_rs.amount.sum(),
         round(naive_rs.amount.sum() / base_gpv, 2)),
        ("naive join to fraud decision", len(naive_fr), naive_fr.amount.sum(),
         round(naive_fr.amount.sum() / base_gpv, 2)),
        ("correct: fraud pre-aggregated", len(good_fr), good_fr.amount.sum(),
         round(good_fr.amount.sum() / base_gpv, 2)),
    ], columns=["method", "rows_returned", "gross_payment_value", "inflation_factor"])

    # why the USD reconciliation breaks - it must decompose to zero residual
    fxs = D["fx_rate"]
    fxs = fxs[fxs.rate_type == "SETTLEMENT"].set_index(["currency", "rate_date"]).rate_to_usd
    j = s.merge(t[["transaction_id", "amount", "currency", "created_at", "status"]],
                on="transaction_id", how="inner")
    key = pd.MultiIndex.from_arrays([j.currency, j.created_at.dt.normalize()])
    expected = j.amount.to_numpy() * fxs.reindex(key).to_numpy()
    actual = j.settlement_amount.to_numpy() + j.fee_amount.to_numpy()
    j["abs_diff"] = np.abs(actual - expected)
    bad = j[j.abs_diff > 0.01]

    dup_ids = set(s.groupby("transaction_id").size().pipe(lambda x: x[x > 1]).index)
    on_dup = bad.transaction_id.isin(dup_ids)
    on_neg = (~on_dup) & (bad.amount < 0)
    residual = len(bad) - int(on_dup.sum()) - int(on_neg.sum())

    out["Reconciliation breakdown"] = pd.DataFrame([
        ("Settlement rows that do not reconcile to transaction value in USD",
         len(bad), ""),
        ("  explained by: transaction has two settlement rows (partial + correction)",
         int(on_dup.sum()), "Defect 'more than one settlement row per transaction'"),
        ("  explained by: transaction carries an impossible negative amount",
         int(on_neg.sum()), "Defect 'amount < 0 with status CAPTURED'"),
        ("  unexplained residual", residual,
         "Zero residual means the break is fully attributed to known defects"),
    ], columns=["item", "rows", "note"])

    return out

details = build_details(D)
for name, df in details.items():
    print(f"\n=== {name} ({len(df)} rows) ===")
    print(df.head(12).to_string(index=False))


=== Missing merchant by channel (4 rows) ===
      channel  attempts  missing_merchant  missing_pct
       WALLET     41053               864        2.105
         CARD    111719               480        0.430
          UPI     71783               296        0.412
BANK_TRANSFER     35732               128        0.358

=== Risk score by model version (2 rows) ===
model_version  decisions  min_score  max_score  scores_above_one  above_one_pct
         v2.1     280293      0.001      0.999                 0           0.00
         v3.0     186158      0.100     99.900            176568          94.85

=== Event lag by month (13 rows) ===
event_month  events  late_over_2d  late_pct  max_lag_days
    2025-01   62436          1931      3.09           9.0
    2025-02   66102          2168      3.28           9.0
    2025-03   69483          2411      3.47           9.0
    2025-04   72889          2511      3.44           9.0
    2025-05   76492          2815      3.68           9.0
    202

## Evidence samples

In [5]:
def build_evidence(D, scorecard, n=10):
    """Sample offending records for each blocking check that actually failed."""
    t, f, s = D["transaction"], D["fraud_decision"], D["settlement"]
    frames = []

    def take(check, df, cols):
        if df.empty:
            return
        x = df.head(n)[cols].copy()
        x.insert(0, "check_name", check)
        x.columns = ["check_name"] + [f"col_{i+1}" for i in range(len(cols))]
        x.insert(1, "columns_shown", ", ".join(cols))
        frames.append(x)

    take("transaction.amount < 0 with status CAPTURED (impossible)",
         t[(t.amount < 0) & (t.status == "CAPTURED")],
         ["transaction_id", "merchant_id", "amount", "currency", "status"])

    take("fraud_decision.risk_score outside 0 to 1",
         f[f.risk_score > 1],
         ["transaction_id", "rule_id", "decision", "risk_score", "model_version"])

    dup_ids = s.groupby("transaction_id").size().pipe(lambda x: x[x > 1]).index[:5]
    take("more than one settlement row per transaction",
         s[s.transaction_id.isin(dup_ids)].sort_values("transaction_id"),
         ["transaction_id", "settlement_amount", "fee_amount", "settlement_status"])

    take("settlement.transaction_id not found in transaction",
         s[~s.transaction_id.isin(set(t.transaction_id))],
         ["transaction_id", "settlement_amount", "fee_amount", "settlement_status"])

    cap = t[t.status == "CAPTURED"]
    take("captured transaction with no settlement row",
         cap[~cap.transaction_id.isin(set(s.transaction_id))],
         ["transaction_id", "merchant_id", "amount", "currency", "created_at"])

    take("transaction.merchant_id is null",
         t[t.merchant_id.isna()],
         ["transaction_id", "customer_id", "channel", "amount", "status"])

    return pd.concat(frames, ignore_index=True)

evidence = build_evidence(D, scorecard)
print(f"{len(evidence)} sample records across {evidence.check_name.nunique()} failing checks")
evidence.head(8)

60 sample records across 6 failing checks


,check_name,columns_shown,col_1,col_2,col_3,col_4,col_5
0,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00000785,MER00540,-90.83,AED,CAPTURED
1,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00002071,MER00323,-3444.78,INR,CAPTURED
2,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00002892,MER00581,-60.95,SGD,CAPTURED
3,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00003008,MER00069,-14347.75,INR,CAPTURED
4,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00003601,NaN,-2234.38,AED,CAPTURED
5,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00005162,MER00323,-3346.85,INR,CAPTURED
6,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00005967,MER00241,-324.83,AED,CAPTURED
7,transaction.amount < 0 with status CAPTURE...,"transaction_id, merchant_id, amount, curre...",TXN00006000,MER00284,-11.13,AED,CAPTURED


## Write the report

In [6]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

NAVY, ARIAL = "1F3864", "Arial"
SEV_FILL = {"Blocking": "F8CBAD", "Warning": "FFE699", "Informational": "E2EFDA"}
RES_FILL = {"FAIL": "F8CBAD", "PASS": "E2EFDA"}
thin = Side(style="thin", color="BFBFBF")
box = Border(left=thin, right=thin, top=thin, bottom=thin)


def write_sheet(wb, title, df, widths=None):
    ws = wb.create_sheet(title[:31])
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    for c in range(1, df.shape[1] + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(name=ARIAL, bold=True, size=10, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor=NAVY)
        cell.alignment = Alignment(vertical="center", wrap_text=True)
        cell.border = box
    ws.row_dimensions[1].height = 30
    for r in range(2, df.shape[0] + 2):
        for c in range(1, df.shape[1] + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = Font(name=ARIAL, size=9)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = box
    for col, w in (widths or {}).items():
        ws.column_dimensions[col].width = w
    ws.freeze_panes = "A2"
    return ws


wb = Workbook()
wb.remove(wb.active)

cover = wb.create_sheet("Cover")
cover.sheet_view.showGridLines = False
lines = [
    ("AstraPay - Payment Profitability Diagnostic", 18, True),
    ("Task 3 - Data Quality Investigation", 13, False),
    ("", 10, False),
    ("Scope: ten source datasets, 1 January 2025 to 31 December 2025", 10, False),
    ("Dimensions: completeness, validity, uniqueness, consistency, timeliness, reconciliation", 10, False),
    ("", 10, False),
    ("Severity vs result", 11, True),
    ("Severity is the potential impact of a check. Result is what happened when it ran.", 10, False),
    ("A Blocking check with result PASS is a control that held, not an open issue.", 10, False),
    ("", 10, False),
    ("Remediation options", 11, True),
    ("Exclude, repair, quarantine, or retain with a flag.", 10, False),
    ("", 10, False),
    ("Reconciliation", 11, True),
    ("The USD settlement mismatch decomposes to a zero residual. See the breakdown sheet.", 10, False),
]
for i, (text, size, bold) in enumerate(lines, start=2):
    c = cover.cell(row=i, column=2, value=text)
    c.font = Font(name=ARIAL, size=size, bold=bold, color=NAVY if bold else "000000")
cover.column_dimensions["A"].width = 3
cover.column_dimensions["B"].width = 110

ws = write_sheet(wb, "DQ Scorecard", scorecard,
                 {"A": 15, "B": 48, "C": 22, "D": 13, "E": 13, "F": 11, "G": 9, "H": 14, "I": 60})
for r in range(2, scorecard.shape[0] + 2):
    res = ws.cell(row=r, column=7).value
    sev = ws.cell(row=r, column=8).value
    if res in RES_FILL:
        ws.cell(row=r, column=7).fill = PatternFill("solid", fgColor=RES_FILL[res])
        ws.cell(row=r, column=7).font = Font(name=ARIAL, size=9, bold=True)
    if sev in SEV_FILL:
        ws.cell(row=r, column=8).fill = PatternFill("solid", fgColor=SEV_FILL[sev])

summary = (scorecard.groupby("severity")
           .agg(checks=("check_name", "count"),
                failing=("result", lambda x: (x == "FAIL").sum()),
                rows_affected=("failed_rows", "sum"))
           .reindex(["Blocking", "Warning", "Informational"]).dropna(how="all").reset_index())
write_sheet(wb, "Severity Summary", summary, {"A": 18, "B": 10, "C": 10, "D": 16})

for name, df in details.items():
    write_sheet(wb, name, df, {"A": 62 if "Reconciliation" in name else 26,
                               "B": 12 if "Reconciliation" in name else 16,
                               "C": 56 if "Reconciliation" in name else 18,
                               "D": 18, "E": 20, "F": 18})

write_sheet(wb, "Evidence samples", evidence,
            {"A": 46, "B": 40, "C": 18, "D": 18, "E": 16, "F": 16, "G": 16})

wb.save(OUT_FILE)
print("wrote", OUT_FILE, "-", len(wb.sheetnames), "sheets")

wrote 03_dq_report.xlsx - 11 sheets


## Sanity check

In [7]:
expect = {
    "transaction.merchant_id is null": (0.5, 1.0),
    "fraud_decision.risk_score outside 0 to 1": (30.0, 45.0),
    "captured transaction with no settlement row": (3.5, 6.0),
    "more than one settlement row per transaction": (0.3, 1.0),
}
idx = scorecard.set_index("check_name")
for name, (lo, hi) in expect.items():
    v = float(idx.loc[name, "failed_pct"])
    print(f"{name:<48} {v:>7.3f}%   expected {lo}-{hi}%   "
          f"{'ok' if lo <= v <= hi else 'CHECK THE DATA'}")

print("\nreconciliation residual must be zero:")
print(details["Reconciliation breakdown"].to_string(index=False))

transaction.merchant_id is null                    0.679%   expected 0.5-1.0%   ok
fraud_decision.risk_score outside 0 to 1          37.853%   expected 30.0-45.0%   ok
captured transaction with no settlement row        4.768%   expected 3.5-6.0%   ok
more than one settlement row per transaction       0.595%   expected 0.3-1.0%   ok

reconciliation residual must be zero:
                                                                      item  rows                                                               note
         Settlement rows that do not reconcile to transaction value in USD  3038                                                                   
  explained by: transaction has two settlement rows (partial + correction)  2668              Defect 'more than one settlement row per transaction'
           explained by: transaction carries an impossible negative amount   370                           Defect 'amount < 0 with status CAPTURED'
                                   